<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/17_working_with_other_languages_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Working with Other Languages

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Most of this course used English text, and most tools work best in English. This notebook is about
everything else: Dutch, which is well served; languages with far less text online, like Limburgish,
Yoruba or Quechua; and corpora that mix several languages at once.

We use **SIB-200** (Adelani et al., 2024): the same 1,004 sentences, translated by professionals into
more than 200 languages, each labeled with one of seven topics. Because the sentences are
**parallel**, every comparison between languages is fair: the content is identical, only the
language changes. That lets us see exactly what a model loses when it moves away from English.

> ⚠️ **Turn on the GPU first:** *Runtime → Change runtime type → T4 GPU.*

In this notebook you will:

- compare how tokenizers treat the same content in nine languages
- fine-tune a Dutch model and a multilingual model on Dutch, against a TF-IDF baseline
- train in English and apply the model to other languages (cross-lingual transfer)
- translate first with NLLB-200, and see when that beats a multilingual model
- identify the languages in a mixed corpus, and check whether a measure means the same thing in every language
- see whether multilingual embeddings group texts by meaning or by language

**Contents**

1. Three routes to other languages
2. The data: one set of sentences, nine languages
3. Tokenizers: the same content, different costs
4. Dutch: a Dutch model or a multilingual one?
5. Train in English, apply everywhere
6. Translate first
7. Low-resource languages
8. A corpus with many languages: which is which?
9. Does the measure mean the same thing everywhere?
10. Multilingual embeddings: meaning or language?
11. Wrap-up

There are 5 exercises along the way, marked ✏️. The solutions notebook has worked answers, but try them yourself first.

## 0. Setup

In [ ]:
!pip install -q transformers accelerate datasets sentence-transformers sentencepiece fasttext lingua-language-detector sacrebleu

import gc
import math

import fasttext
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sacrebleu
import torch
from datasets import Dataset
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

SEED = 42
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 140)
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "none: Runtime → Change runtime type → T4 GPU")

---
## 1. Three routes to other languages

| Route | What you do | Works well when | Watch out for |
|---|---|---|---|
| **A model for your language** | a monolingual BERT, e.g. RobBERT or BERTje for Dutch | one exists and was trained on lots of text | only a few dozen languages have a good one |
| **A multilingual model** | one model for many languages: XLM-RoBERTa, mDeBERTa, mmBERT | your languages were in its training data | capacity is shared, so each language gets less |
| **Translate first** | machine translation into English, then English tools | translation quality is good for your language | nuance gets lost; texts leave your machine with commercial services |

LLMs are a fourth route, with the same caveat as multilingual models, only stronger: they can sound
fluent in a language they handle badly. Whatever the route, the rule from the rest of the course
holds, with one addition: **validate in every language separately.**

---
## 2. The data: one set of sentences, nine languages

SIB-200 sentences come from Wikinews, Wikijunior and Wikivoyage, translated for the Flores-200
benchmark. Each language has the same 701 training, 99 validation and 204 test sentences. We pick
nine languages from very well served to barely served:

- **English, Dutch, German**: lots of text, good tools;
- **Limburgish, Luxembourgish**: small regional languages, close relatives of Dutch and German;
- **Swahili, Yoruba, Amharic** (in its own Ge'ez script), **Ayacucho Quechua**: spoken by millions,
  but with far less digital text.

In [ ]:
LANGS = {"eng_Latn": "English", "nld_Latn": "Dutch", "deu_Latn": "German", "lim_Latn": "Limburgish",
         "ltz_Latn": "Luxembourgish", "swh_Latn": "Swahili", "yor_Latn": "Yoruba", "amh_Ethi": "Amharic",
         "quy_Latn": "Quechua"}


def load_sib(lang):
    parts = {}
    for split in ["train", "validation", "test"]:
        path = hf_hub_download(repo_id="Davlan/sib200", repo_type="dataset", revision="refs/convert/parquet",
                               filename=f"{lang}/{split}/0000.parquet")
        df = pd.read_parquet(path)
        parts[split] = (df.sort_values("index_id") if "index_id" in df else df).reset_index(drop=True)
    return parts


data = {lang: load_sib(lang) for lang in LANGS}
for lang in LANGS:                                       # parallel: same topics in the same order
    assert (data[lang]["test"].category.values == data["eng_Latn"]["test"].category.values).all(), lang

CATEGORIES = sorted(data["eng_Latn"]["train"].category.unique())
print({split: len(df) for split, df in data["eng_Latn"].items()}, "\n", CATEGORIES)
data["eng_Latn"]["train"].category.value_counts()

In [ ]:
k = 3
print("Topic:", data["eng_Latn"]["test"].category[k], "\n")
for lang, name in LANGS.items():
    print(f"{name:<14} {data[lang]['test'].text[k]}")

---
## 3. Tokenizers: the same content, different costs

Every transformer starts by cutting text into tokens (Day 3). A tokenizer built mostly from English
text cuts other languages into many more, smaller pieces. Because our sentences are parallel, we can
measure that directly: **how many tokens does each language need for the same 204 sentences,
relative to English?** A value of 2 means twice as many tokens: twice the cost with a paid API, a
longer wait, and less text fitting into the model at once (Petrov et al., 2023).

In [ ]:
TOKENIZERS = {"DistilBERT (English)": "distilbert/distilbert-base-uncased",
              "RobBERT (Dutch)": "pdelobelle/robbert-v2-dutch-base",
              "XLM-R (100 languages)": "FacebookAI/xlm-roberta-base",
              "mmBERT (1,800+ languages)": "jhu-clsp/mmBERT-base",
              "Qwen2.5 (an LLM)": "Qwen/Qwen2.5-1.5B-Instruct"}
toks = {name: AutoTokenizer.from_pretrained(ckpt) for name, ckpt in TOKENIZERS.items()}

counts = pd.DataFrame({name: {LANGS[lang]: sum(len(tok.tokenize(t)) for t in data[lang]["test"].text)
                              for lang in LANGS} for name, tok in toks.items()})
relative = counts / counts.loc["English"]
relative.round(2).style.background_gradient(cmap="Reds", axis=None)

In [ ]:
unk = toks["DistilBERT (English)"].unk_token
amharic = data["amh_Ethi"]["test"].text[k]
print("DistilBERT:", toks["DistilBERT (English)"].tokenize(amharic)[:12])
print("XLM-R:     ", toks["XLM-R (100 languages)"].tokenize(amharic)[:12])
yoruba = data["yor_Latn"]["test"].text[k]
print("\nYoruba, XLM-R:", toks["XLM-R (100 languages)"].tokenize(yoruba)[:15])

Read the table by column. The English model is cheap for English and expensive for everything else,
and for a script it has never seen, like Ge'ez, it may produce `[UNK]` (unknown) tokens: the text is
simply gone. Multilingual tokenizers spread the cost more evenly, but rarely evenly. A quick
diagnostic for your own data: tokenize a few sentences and look at the pieces. If words shatter into
single letters, the model knows the language poorly.

> **✏️ Exercise 1**
>
> Pick one more language from SIB-200, ideally one you know: for example `fra_Latn` (French),
> `tur_Latn` (Turkish), `zho_Hans` (Chinese) or `arb_Arab` (Arabic). Load it with `load_sib`, add its
> tokens-relative-to-English to the table, and print how XLM-R and Qwen2.5 split one of its sentences.
> Do the pieces make sense to you?

In [ ]:
# Your code here

---
## 4. Dutch: a Dutch model or a multilingual one?

Three classifiers for Dutch, all trained on the 701 Dutch training sentences and tested on the same
204 test sentences:

- **TF-IDF + logistic regression**: the baseline from Day 1;
- **RobBERT** (Delobelle et al., 2020): a RoBERTa model trained on Dutch only;
- **XLM-RoBERTa** (Conneau et al., 2020): one model for 100 languages.

The helpers below follow the Day 3 BERT notebook. After training we keep only the model, which frees
the GPU memory of the optimizer.

In [ ]:
label2id = {c: i for i, c in enumerate(CATEGORIES)}
id2label = {i: c for c, i in label2id.items()}


def to_dataset(df, tok):
    ds = Dataset.from_dict({"text": df["text"].tolist(), "label": df["category"].map(label2id).tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)


def fine_tune(checkpoint, train_df, epochs=10, learning_rate=2e-5, batch_size=16, seed=SEED):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=len(CATEGORIES), id2label=id2label, label2id=label2id)
    warmup = int(0.1 * math.ceil(len(train_df) / batch_size) * epochs)
    args = TrainingArguments(output_dir="tmp-run", num_train_epochs=epochs, learning_rate=learning_rate,
                             per_device_train_batch_size=batch_size, weight_decay=0.01, warmup_steps=warmup,
                             save_strategy="no", logging_strategy="no", seed=seed,
                             fp16=(device == "cuda"), report_to="none")
    trainer = Trainer(model=model, args=args, train_dataset=to_dataset(train_df, tok),
                      data_collator=DataCollatorWithPadding(tokenizer=tok))
    trainer.train()
    model = trainer.model.eval()
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    return model, tok


@torch.no_grad()
def predict(classifier, texts, batch_size=64):
    model, tok = classifier
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(list(texts[i:i + batch_size]), truncation=True, max_length=128, padding=True,
                  return_tensors="pt").to(model.device)
        out.append(model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits.argmax(-1).cpu().numpy())
    return np.array(CATEGORIES)[np.concatenate(out)]


def scores(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro-F1": f1_score(y_true, y_pred, labels=CATEGORIES, average="macro", zero_division=0)}

In [ ]:
nl_train, nl_test = data["nld_Latn"]["train"], data["nld_Latn"]["test"]
results = {}

tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
logreg = LogisticRegression(max_iter=2000).fit(tfidf.fit_transform(nl_train.text), nl_train.category)
results["TF-IDF + logistic regression"] = scores(nl_test.category, logreg.predict(tfidf.transform(nl_test.text)))

robbert_nl = fine_tune("pdelobelle/robbert-v2-dutch-base", nl_train)
results["RobBERT (Dutch)"] = scores(nl_test.category, predict(robbert_nl, nl_test.text.tolist()))

xlmr_nl = fine_tune("FacebookAI/xlm-roberta-base", nl_train)
results["XLM-R (multilingual)"] = scores(nl_test.category, predict(xlmr_nl, nl_test.text.tolist()))

pd.DataFrame(results).T.round(3)

With only 701 training sentences, differences of a few points are within the noise of a single run
(Day 3 showed how much seeds matter). The usual pattern in larger studies: a good monolingual model
is at least as good as a multilingual one **for its own language**, because a multilingual model
shares its capacity across all its languages, the "curse of multilinguality" (Conneau et al., 2020).
What the multilingual model offers is the next section.

---
## 5. Train in English, apply everywhere

A multilingual model maps sentences in different languages to similar representations. So you can
fine-tune it on **English** labels and apply it to languages it never saw labels for:
**zero-shot cross-lingual transfer**. Because the test sets are parallel, any difference between
languages below is purely the model's ability to read that language.

In [ ]:
xlmr_en = fine_tune("FacebookAI/xlm-roberta-base", data["eng_Latn"]["train"])

transfer = pd.DataFrame({LANGS[lang]: scores(data[lang]["test"].category, predict(xlmr_en, data[lang]["test"].text.tolist()))
                         for lang in LANGS}).T
transfer["tokens vs. English (XLM-R)"] = relative["XLM-R (100 languages)"]
transfer.round(2)

In [ ]:
ax = transfer["accuracy"].plot(kind="barh", color="#34B233", figsize=(7, 4))
ax.axvline(1 / len(CATEGORIES), color="gray", linestyle="--", label="guessing")
ax.invert_yaxis()
ax.set_xlabel("Accuracy on the same 204 sentences, trained on English only")
ax.legend()
plt.tight_layout()
plt.show()

Expect a staircase: close to English for the big European languages, lower for the regional ones,
and much lower for languages the model saw little or nothing of in pretraining (check the XLM-R
paper for its list of 100 languages). In the full SIB-200 study, transfer from English reached less
than 70% accuracy for 81 of 205 languages, with languages from Africa, the Americas and Oceania doing
worst (Adelani et al., 2024). That is a measurement problem, and an equity problem: the speakers of
those languages are the ones whose texts get measured worst.

> **✏️ Exercise 2**
>
> Transfer doesn't have to start from English. Evaluate the XLM-R model you trained on **Dutch**
> (`xlmr_nl`, Section 4) on all nine languages, and put its accuracy next to the English-trained one.
> Which languages gain? Why might that be?

In [ ]:
# Your code here

---
## 6. Translate first

The other route: translate everything into English and use an English model. **NLLB-200** (NLLB Team,
2024) is an open translation model for 200 languages that runs on Colab's GPU, so the texts never
leave your machine. Its license allows research use only, not commercial use.

Because SIB-200 is parallel, we also know the correct English for every sentence, so we can score the
translations themselves with **chrF** (character overlap with the reference, 0 to 100; higher is
better).

In [ ]:
NLLB = "facebook/nllb-200-distilled-600M"
nllb = AutoModelForSeq2SeqLM.from_pretrained(NLLB, dtype=torch.float16 if device == "cuda" else torch.float32).to(device)


@torch.no_grad()
def translate(texts, src, tgt="eng_Latn", batch_size=32):
    nllb_tok = AutoTokenizer.from_pretrained(NLLB, src_lang=src)     # the tokenizer marks the source language
    out = []
    for i in range(0, len(texts), batch_size):
        enc = nllb_tok(list(texts[i:i + batch_size]), return_tensors="pt", padding=True, truncation=True,
                       max_length=256).to(device)
        generated = nllb.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                                  forced_bos_token_id=nllb_tok.convert_tokens_to_ids(tgt), max_new_tokens=256)
        out += nllb_tok.batch_decode(generated, skip_special_tokens=True)
    return out


english_test = data["eng_Latn"]["test"].text.tolist()
translated, rows = {}, []
for lang in LANGS:
    if lang == "eng_Latn":
        continue
    translated[lang] = translate(data[lang]["test"].text.tolist(), src=lang)
    rows.append({"language": LANGS[lang],
                 "chrF of the translation": sacrebleu.corpus_chrf(translated[lang], [english_test]).score,
                 "direct (Section 5)": transfer.loc[LANGS[lang], "accuracy"],
                 "translated first": accuracy_score(data[lang]["test"].category, predict(xlmr_en, translated[lang]))})
translate_first = pd.DataFrame(rows).set_index("language")
translate_first.round(2)

In [ ]:
for lang in ["lim_Latn", "yor_Latn", "quy_Latn"]:
    print(f"{LANGS[lang]:<11} {translated[lang][k]}")
print(f"{'Reference':<11} {english_test[k]}")

Translation helps most where the classifier reads the language badly **and** the translation model
reads it well; where translation quality (chrF) is low, translating first only moves the problem. For
bag-of-words methods, machine translation into English has been shown to work well for comparative
research in major European languages (de Vries et al., 2018). Check it for yours: translate a sample
and have a native speaker read it.

---
## 7. Low-resource languages

"Low-resource" means little digital text, few labeled datasets, and weak tools, and it covers most of
the world's languages, including many spoken by tens of millions of people. Dialects, non-standard
spelling and languages written in another script (Arabic in Latin letters, for example) behave like
low-resource languages even inside big ones. What helps:

1. **Check whether the model saw your language** in pretraining (the model card or paper lists the
   languages), and run the tokenizer check from Section 3. Regional models, such as AfroXLMR for
   African languages or IndicBERT for languages of India, often beat global ones in their region.
2. **Continue pretraining on your own unlabeled texts** before fine-tuning (Gururangan et al., 2020).
   It is cheap compared to labeling, and it teaches the model your language and your domain.
3. **Transfer from a related language** (Exercise 2) rather than from English.
4. **Translate the training data** into your language (Exercise 3), or your texts into English
   (Section 6), and have native speakers check samples of the translation.
5. **Be extra careful with LLMs.** They are much weaker in low-resource languages, but still fluent,
   so their mistakes are harder to spot.
6. **Work with native speakers** on the codebook, the annotation and the error analysis. Communities
   such as Masakhane, for African languages, build data and models together with speakers.

> **✏️ Exercise 3**
>
> **Translate-train.** Translate the 701 English training sentences into Yoruba with
> `translate(..., src="eng_Latn", tgt="yor_Latn")`, fine-tune XLM-R on the translations, and test on the
> real Yoruba test set. Compare with zero-shot transfer (Section 5) and translate-first (Section 6).

In [ ]:
# Your code here

---
## 8. A corpus with many languages: which is which?

Now a corpus like you might scrape from social media or a multilingual parliament: all 9 × 204 test
sentences, shuffled, and we pretend we don't know their languages. Step one is always **language
identification**. We compare two tools:

- **Lingua**: accurate, also on short texts, for 75 widely used languages;
- **GlotLID** (Kargaran et al., 2023): a fastText model for over 2,000 language labels, built for
  low-resource languages.

In [ ]:
mixed = (pd.concat([data[lang]["test"].assign(language=lang, sid=range(len(data[lang]["test"]))) for lang in LANGS])
           .sample(frac=1, random_state=SEED).reset_index(drop=True))
print(len(mixed), "sentences")

from lingua import LanguageDetectorBuilder

lingua_detector = LanguageDetectorBuilder.from_all_languages().build()
LINGUA_CODES = {"swa": "swh"}           # Lingua uses the code for Swahili as a whole


def lingua_id(text):
    found = lingua_detector.detect_language_of(text)
    code = found.iso_code_639_3.name.lower() if found else "unknown"
    return LINGUA_CODES.get(code, code)


glotlid = fasttext.load_model(hf_hub_download(repo_id="cis-lmu/glotlid", filename="model.bin"))


def glotlid_id(texts):
    labels, _ = glotlid.predict([t.replace("\n", " ") for t in texts], k=1)   # a list avoids a NumPy 2 issue
    return [lab[0].replace("__label__", "") for lab in labels]


mixed["lingua"] = [lingua_id(t) for t in mixed.text]
mixed["glotlid"] = glotlid_id(mixed.text)
mixed["true"] = mixed.language
lid = pd.DataFrame({"Lingua": (mixed.lingua == mixed.true.str[:3]).groupby(mixed.true.map(LANGS)).mean(),
                    "GlotLID": (mixed.glotlid == mixed.true).groupby(mixed.true.map(LANGS)).mean()})
lid.loc[list(LANGS.values())].round(2)

In [ ]:
for lang in ["lim_Latn", "ltz_Latn", "quy_Latn"]:
    print(f"{LANGS[lang]} according to Lingua:", mixed[mixed.true == lang].lingua.value_counts().head(3).to_dict())

A tool can only recognize languages it knows. Lingua doesn't know Limburgish or Quechua, so it files
them under something it does know, without telling you. With a big tool like GlotLID the risk shifts
to confusing close relatives. Either way: check a sample of every language your tool reports, keep an
"uncertain" category for low-confidence cases, and identify the language **per sentence** if your
texts switch languages midway (code-switching).

> **✏️ Exercise 4**
>
> Social media texts are short. Repeat the language identification on only the **first five words**
> of each sentence. Which languages suffer most, and with which tool?

In [ ]:
# Your code here

---
## 9. Does the measure mean the same thing everywhere?

In comparative research the question is rarely "how accurate is the model?" but "is country A really
different from country B?". Here we can answer that exactly. The corpus is parallel, so the **true**
share of each topic is identical in every language. Any difference in the **predicted** shares is
pure measurement error, which in a real study would look like a finding.

In [ ]:
mixed["predicted"] = predict(xlmr_en, mixed.text.tolist())
true_share = data["eng_Latn"]["test"].category.value_counts(normalize=True)
predicted_share = pd.crosstab(mixed.language.map(LANGS), mixed.predicted, normalize="index").reindex(columns=CATEGORIES, fill_value=0)

topic = "politics"
ax = predicted_share.loc[list(LANGS.values()), topic].plot(kind="barh", color="#34B233", figsize=(7, 4))
ax.axvline(true_share[topic], color="black", linestyle="--", label="true share (same in every language)")
ax.invert_yaxis()
ax.set_xlabel(f"Predicted share of '{topic}'")
ax.legend()
plt.tight_layout()
plt.show()

If you had compared "how much politics" across these nine languages, part of the differences would
come from the model, not from the texts. In a real study you can't see this directly, so you have to
build the check in:

- a **validation sample in every language**, coded by native speakers with the same codebook;
- **accuracy per language** in your paper, not one overall number;
- differences between groups that are **larger than the differences in accuracy** between their
  languages, before you interpret them.

> **✏️ Exercise 5**
>
> Measure the distortion for all seven topics at once: for each language, the **total variation
> distance** between predicted and true topic shares (half the sum of the absolute differences; 0 is
> perfect). Do the same for the translate-first predictions from Section 6. Which strategy gives more
> comparable measures across languages?

In [ ]:
# Your code here

---
## 10. Multilingual embeddings: meaning or language?

Sentence embeddings (Thursday's session, BERTopic) can be multilingual too. In an ideal multilingual
space, a sentence sits closest to **its own translations**. In practice, embeddings often group texts
by **language** first, and then a topic model finds "the Swahili topic" instead of "the health topic".
For every sentence in the mixed corpus we find its nearest neighbor and ask: is it a translation of the
same sentence, another sentence in the same language, or something else?

In [ ]:
def nearest_neighbors(model_name):
    emb = SentenceTransformer(model_name, device=device).encode(mixed.text.tolist(), batch_size=64,
                                                               normalize_embeddings=True, show_progress_bar=True)
    sims = emb @ emb.T
    np.fill_diagonal(sims, -1)
    nn = sims.argmax(axis=1)
    same_sentence = mixed.sid.to_numpy()[nn] == mixed.sid.to_numpy()
    same_language = mixed.language.to_numpy()[nn] == mixed.language.to_numpy()
    kind = np.where(same_sentence, "its translation", np.where(same_language, "same language", "other"))
    return pd.Series(kind).groupby(mixed.language.map(LANGS)).value_counts(normalize=True).unstack(fill_value=0)


neighbors = {name: nearest_neighbors(ckpt) for name, ckpt in
             {"multilingual MiniLM": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
              "LaBSE": "sentence-transformers/LaBSE"}.items()}
pd.concat(neighbors, axis=1).loc[list(LANGS.values())].round(2)

LaBSE (Feng et al., 2022) was trained specifically to put translations together, and usually does so
for most languages; general multilingual models do it well for big languages and less so for the
rest. Before running a topic model on a multilingual corpus, run this kind of check, and afterwards
look at the language mix of every topic: a topic that is 95% one language is probably about the
language, not the content. Translating first, or embeddings that align languages well, are the usual
fixes (Lind et al., 2022; Maier et al., 2022).

---
## 11. Wrap-up

A checklist for text analysis beyond English:

- **Know your languages.** Identify the language of every text (per sentence if they mix), and check a
  sample of what your tool reports.
- **Check the model's languages**: the pretraining list, and the tokenizer on your own texts.
- **Choose a route** (a model for your language, a multilingual model, translation first) and compare
  them on your validation data rather than assuming.
- **Validate in every language**, with native-speaker coders and the same codebook, and report
  accuracy per language.
- **Compare with care.** Differences between languages or countries must be larger than the
  differences in measurement quality between them.
- **For low-resource languages**, combine strategies: related-language transfer, continued
  pretraining, translated training data, and above all people who speak the language.

### Further reading

- Adelani, D. I., Liu, H., Shen, X., Vassilyev, N., Alabi, J. O., Mao, Y., Gao, H., & Lee, E.-S. A.
  (2024). SIB-200: A simple, inclusive, and big evaluation dataset for topic classification in 200+
  languages and dialects. *Proceedings of EACL 2024*.
- Conneau, A., et al. (2020). Unsupervised cross-lingual representation learning at scale.
  *Proceedings of ACL 2020*. (XLM-RoBERTa.)
- Delobelle, P., Winters, T., & Berendt, B. (2020). RobBERT: A Dutch RoBERTa-based language model.
  *Findings of EMNLP 2020*.
- de Vries, E., Schoonvelde, M., & Schumacher, G. (2018). No longer lost in translation: Evidence that
  Google Translate works for comparative bag-of-words text applications. *Political Analysis, 26*(4).
- Feng, F., Yang, Y., Cer, D., Arivazhagan, N., & Wang, W. (2022). Language-agnostic BERT sentence
  embedding. *Proceedings of ACL 2022*. (LaBSE.)
- Gururangan, S., et al. (2020). Don't stop pretraining: Adapt language models to domains and tasks.
  *Proceedings of ACL 2020*.
- Kargaran, A. H., Imani, A., Yvon, F., & Schütze, H. (2023). GlotLID: Language identification for
  low-resource languages. *Findings of EMNLP 2023*.
- Licht, H. (2023). Cross-lingual classification of political texts using multilingual sentence
  embeddings. *Political Analysis, 31*(3).
- Lind, F., et al. (2022). Building the bridge: Topic modeling for comparative research.
  *Communication Methods and Measures, 16*(2).
- Maier, D., et al. (2022). Machine translation vs. multilingual dictionaries: Assessing two strategies
  for the topic modeling of multilingual text collections. *Communication Methods and Measures, 16*(1).
- Marone, M., et al. (2025). mmBERT: A modern multilingual encoder with annealed language learning.
  *arXiv:2509.06888*.
- NLLB Team (2024). Scaling neural machine translation to 200 languages. *Nature, 630*, 841–846.
- Petrov, A., La Malfa, E., Torr, P., & Bibi, A. (2023). Language model tokenizers introduce unfairness
  between languages. *NeurIPS 2023*.